# NB26 — PAH Final: Multi-Seed Doğrulama + Feature Engineering + Hyper-Ensemble

PAH panelinde Boot %80/20 F1 ≈ 0.58 platosuna ulaşıldı (NB21-NB25).
Bu notebook üç ekseni birleştirir:

1. **Multi-seed doğrulama** — Tek seed'e bağımlılığı kaldır (10 farklı seed ile tekrarlı split)
2. **Feature Engineering** — Anonimleştirilmiş veriyle yapılabilecek düşük riskli FE
3. **Hyper-Ensemble** — Partition-based undersampling + LGBM ensemble (parSMURF konsepti)

Her biri bağımsız senaryo olarak değerlendirilerek en güçlü kombinasyonlar tespit edilir.

## Deney Matrisi

| # | Senaryo | Eğitim | FE | Model | Açıklama |
|---|---------|--------|-----|-------|----------|
| S1 | Baseline (NB21 repro) | COMBINED (3430) | Yok | BalBag(20)+LGBM | NB21 P4 reproduksiyon |
| S2 | + Feature Engineering | COMBINED | FE | BalBag(20)+LGBM | FE katkısı ölçümü |
| S3 | + Hyper-Ensemble | COMBINED | Yok | HyperEns(k=5)+LGBM | parSMURF konsepti |
| S4 | FE + Hyper-Ensemble | COMBINED | FE | HyperEns(k=5)+LGBM | İkisi birlikte |
| S5 | MASTER+PAH-half FE | MASTER+PAH-train | FE | BalBag(20)+LGBM | NB25 E2 + FE |
| S6 | Sweep n=10 + FE | COMBINED | FE | BalBag(10, mf=1.0)+LGBM | NB22 en iyi config + FE |

## Referanslar:
- NB21 P4: Boot=0.582, MCC=0.529 (%80/20)
- NB22 sweep n=10: Boot=0.591, MCC=0.536
- NB25 E2 (MASTER+PAH): MCC=0.5694


In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

sys.path.insert(0, os.path.abspath(".."))
from config import SEED, PROJECT_ROOT
import src.columns_real as CR

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    matthews_corrcoef, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix
)
from lightgbm import LGBMClassifier
from imblearn.ensemble import BalancedBaggingClassifier

np.random.seed(SEED)

# Sabitler
ID_COL = CR.ID_COL
TARGET = CR.TARGET_COL
PI_TEST = 0.20
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = SEED
HIGH_MISS_THR = 0.50

# Multi-seed listesi (10 farklı seed)
MULTI_SEEDS = [42, 123, 456, 789, 1024, 2048, 3141, 4242, 5555, 9999]

# Sonuc dizini
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v11_pah_final")
REPORTS_DIR_NB = os.path.join(PROJECT_ROOT, "reports")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR_NB, exist_ok=True)

print(f"NB26 -- PAH Final Experiments")
print(f"SEED={SEED}, PI_TEST={PI_TEST}, N_BOOT={N_BOOT}")
print(f"Multi-seeds: {MULTI_SEEDS}")
print(f"Results -> {RESULTS_DIR}")


NB26 -- PAH Final Experiments
SEED=42, PI_TEST=0.2, N_BOOT=50
Multi-seeds: [42, 123, 456, 789, 1024, 2048, 3141, 4242, 5555, 9999]
Results -> /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final


In [2]:
# Cell 2: Veri Yukleme + Sutun Temizligi
data_dir = os.path.join(PROJECT_ROOT, "data", "real_data")
df_master = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_MASTER.csv"))
df_kanser = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_KANSER.csv"))
df_cftr   = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_CFTR.csv"))
df_pah    = pd.read_csv(os.path.join(data_dir, "YARISMA_TRAIN_PAH.csv"))

print(f"Ilk boyutlar:")
print(f"  MASTER: {df_master.shape} (pos={df_master[TARGET].sum()}, neg={(df_master[TARGET]==0).sum()})")
print(f"  KANSER: {df_kanser.shape} (pos={df_kanser[TARGET].sum()}, neg={(df_kanser[TARGET]==0).sum()})")
print(f"  CFTR:   {df_cftr.shape}   (pos={df_cftr[TARGET].sum()}, neg={(df_cftr[TARGET]==0).sum()})")
print(f"  PAH:    {df_pah.shape}  (pos={df_pah[TARGET].sum()}, neg={(df_pah[TARGET]==0).sum()})")

# COMBINED: MASTER + KANSER + CFTR (PAH HARIC)
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)
print(f"\nCOMBINED (MASTER+KANSER+CFTR): {df_combined.shape}")

# Cross-panel exact-dup drop
feat_cols = [c for c in df_pah.columns if c not in [ID_COL, TARGET]]

def find_exact_dups(panel_df, master_df, feat_cols, target):
    common_ids = set(panel_df[ID_COL]) & set(master_df[ID_COL])
    if not common_ids:
        return []
    dup_ids = []
    check_cols = feat_cols + [target]
    for vid in common_ids:
        p_rows = panel_df.loc[panel_df[ID_COL] == vid, check_cols]
        if len(p_rows) == 0:
            continue
        p_row = p_rows.iloc[0]
        m_rows = master_df.loc[master_df[ID_COL] == vid, check_cols]
        for _, m_row in m_rows.iterrows():
            if p_row.equals(m_row):
                dup_ids.append(vid)
                break
    return dup_ids

dup_ids = find_exact_dups(df_pah, df_master, feat_cols, TARGET)
if dup_ids:
    df_pah = df_pah[~df_pah[ID_COL].isin(dup_ids)].reset_index(drop=True)
    print(f"PAH: {len(dup_ids)} birebir-ayni satir drop edildi -> {df_pah.shape}")
else:
    print(f"PAH: birebir-ayni satir yok")

# Sutun temizligi (MASTER uzerinde tespit)
constant_cols = [c for c in feat_cols if df_master[c].nunique(dropna=False) <= 1]

def get_duplicate_col_pairs(df, cols):
    dup_pairs = []
    seen = set()
    for i, c1 in enumerate(cols):
        if c1 in seen:
            continue
        for c2 in cols[i+1:]:
            if c2 in seen:
                continue
            if df[c1].equals(df[c2]):
                dup_pairs.append((c1, c2))
                seen.add(c2)
    return dup_pairs, seen

num_feat = [c for c in feat_cols if c not in [ID_COL, TARGET]]
dup_pairs, dup_drop = get_duplicate_col_pairs(df_master, num_feat)
drop_cols = set(constant_cols) | dup_drop
print(f"\nSutun temizligi: Constant={len(constant_cols)}, Dup pairs={len(dup_pairs)}, Drop={len(drop_cols)}")

keep_cols = [c for c in feat_cols if c not in drop_cols]
df_master = df_master[[ID_COL, TARGET] + keep_cols].copy()
df_kanser = df_kanser[[ID_COL, TARGET] + keep_cols].copy()
df_cftr = df_cftr[[ID_COL, TARGET] + keep_cols].copy()
df_pah = df_pah[[ID_COL, TARGET] + keep_cols].copy()
df_combined = pd.concat([df_master, df_kanser, df_cftr], ignore_index=True)

print(f"\nFinal: MASTER={df_master.shape}, COMBINED={df_combined.shape}, PAH={df_pah.shape}")


Ilk boyutlar:
  MASTER: (2931, 353) (pos=2149, neg=782)
  KANSER: (388, 353) (pos=268, neg=120)
  CFTR:   (111, 353)   (pos=90, neg=21)
  PAH:    (372, 353)  (pos=310, neg=62)

COMBINED (MASTER+KANSER+CFTR): (3430, 353)
PAH: 3 birebir-ayni satir drop edildi -> (369, 353)

Sutun temizligi: Constant=0, Dup pairs=58, Drop=58

Final: MASTER=(2931, 295), COMBINED=(3430, 295), PAH=(369, 295)


In [3]:
# Cell 3: Feature Engineering
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# === Grantham (1974) mesafe matrisi ===
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}

def grantham(a, b):
    if a == b:
        return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a)) or -1

# === BLOSUM62 ===
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62_RAW = """A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4"""
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)
        _B62[(row_aa, col_aa)] = v
        _B62[(col_aa, row_aa)] = v

def blosum62(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    """Anonim-safe feature engineering. Satir-bazli, fit gerektirmez."""
    out = df.copy()
    a1 = out["AA_1"].astype("object").fillna("X")
    a2 = out["AA_2"].astype("object").fillna("X")
    
    # 1. Stop-gain flag (AA_2 == '*')
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    
    # 2. Non-standard AA flag
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    
    # 3. AA same flag (synonymous-like)
    out["fe_aa_same"] = (a1 == a2).astype(int)
    
    # 4. Grantham distance
    out["fe_grantham"] = [grantham(x, y) if (isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA) else -1 for x, y in zip(a1, a2)]
    out["fe_grantham"] = out["fe_grantham"].astype(float)
    
    # 5. BLOSUM62 score
    out["fe_blosum62"] = [blosum62(x, y) if (isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA) else 0 for x, y in zip(a1, a2)]
    out["fe_blosum62"] = out["fe_blosum62"].astype(float)
    
    # 6. Row-wise missing count (tum AL + EK sutunlari)
    al_ek_cols = [c for c in out.columns if c.startswith("AL_") or c.startswith("EK_")]
    out["fe_missing_count"] = out[al_ek_cols].isnull().sum(axis=1).astype(int)
    out["fe_missing_frac"] = out[al_ek_cols].isnull().mean(axis=1).astype(float)
    
    # 7. EK score aggregations (EK_1..EK_9)
    ek_cols = [c for c in out.columns if c.startswith("EK_")]
    if len(ek_cols) > 0:
        out["fe_ek_mean"] = out[ek_cols].mean(axis=1)
        out["fe_ek_std"] = out[ek_cols].std(axis=1)
        out["fe_ek_min"] = out[ek_cols].min(axis=1)
        out["fe_ek_max"] = out[ek_cols].max(axis=1)
        out["fe_ek_range"] = out["fe_ek_max"] - out["fe_ek_min"]
    
    # 8. Log-transform genis aralikli AL frekans sutunlari
    for c in out.columns:
        if c.startswith("AL_") and out[c].dtype in [np.float64, np.float32, float]:
            col_max = out[c].max()
            if col_max > 1.0 and not np.isnan(col_max):
                out[f"fe_{c}_log"] = np.log1p(out[c].clip(lower=0))
    
    return out

FE_NEW_COLS_BASE = [
    "fe_aa_stopgain", "fe_aa_nonstandard", "fe_aa_same",
    "fe_grantham", "fe_blosum62",
    "fe_missing_count", "fe_missing_frac",
    "fe_ek_mean", "fe_ek_std", "fe_ek_min", "fe_ek_max", "fe_ek_range"
]

# Test FE
_test_fe = add_fe(df_pah.head(5))
fe_log_cols = [c for c in _test_fe.columns if c.startswith("fe_") and c.endswith("_log")]
FE_ALL_COLS = FE_NEW_COLS_BASE + fe_log_cols
print(f"FE: {len(FE_NEW_COLS_BASE)} base + {len(fe_log_cols)} log = {len(FE_ALL_COLS)} toplam yeni sutun")
print(f"Ornek FE sutunlari: {FE_ALL_COLS[:8]}")

# Validation: check stopgain count in PAH
_full_fe = add_fe(df_pah)
print(f"\nPAH FE istatistikleri:")
print(f"  stopgain: {_full_fe['fe_aa_stopgain'].sum()}")
print(f"  nonstandard: {_full_fe['fe_aa_nonstandard'].sum()}")
print(f"  aa_same: {_full_fe['fe_aa_same'].sum()}")
print(f"  grantham ornek: {_full_fe['fe_grantham'].head(5).tolist()}")
print(f"  missing_count ornek: {_full_fe['fe_missing_count'].describe().to_dict()}")


FE: 12 base + 1 log = 13 toplam yeni sutun
Ornek FE sutunlari: ['fe_aa_stopgain', 'fe_aa_nonstandard', 'fe_aa_same', 'fe_grantham', 'fe_blosum62', 'fe_missing_count', 'fe_missing_frac', 'fe_ek_mean']

PAH FE istatistikleri:
  stopgain: 0
  nonstandard: 7
  aa_same: 7
  grantham ornek: [22.0, 29.0, 98.0, 102.0, 74.0]
  missing_count ornek: {'count': 369.0, 'mean': 159.17344173441734, 'std': 99.24764398322058, 'min': 0.0, '25%': 52.0, '50%': 177.0, '75%': 268.0, 'max': 286.0}


In [4]:
# Cell 4: Preprocessing (M3 + FE-aware)

def fit_preprocessor(train_df, keep_cols, target, high_miss_thr=HIGH_MISS_THR):
    X = train_df[keep_cols].copy()
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    miss_frac = X[num_cols].isnull().mean()
    high_miss = miss_frac[miss_frac > high_miss_thr].index.tolist()
    medians = X[num_cols].median()
    le_maps = {}
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = LabelEncoder()
        le.fit(X[c])
        le_maps[c] = le
    return {
        "cat_cols": cat_cols, "num_cols": num_cols,
        "high_miss": high_miss, "medians": medians,
        "le_maps": le_maps
    }

def transform_X(df, keep_cols, prep):
    X = df[keep_cols].copy()
    cat_cols = prep["cat_cols"]
    num_cols = prep["num_cols"]
    for c in prep["high_miss"]:
        X[f"is_missing_{c}"] = X[c].isnull().astype(int)
    for c in num_cols:
        X[c] = X[c].fillna(prep["medians"][c])
    for c in cat_cols:
        X[c] = X[c].fillna("MISSING")
        le = prep["le_maps"][c]
        X[c] = X[c].apply(lambda v: le.transform([v])[0] if v in le.classes_ else -1)
    return X

print("Preprocessing pipeline hazir.")

Preprocessing pipeline hazir.


In [5]:
# Cell 5: Evaluation Altyapisi

def adjust_prior_shift(proba, pi_train, pi_test=PI_TEST):
    proba = np.clip(proba, 1e-7, 1 - 1e-7)
    odds = proba / (1 - proba)
    R = (pi_test / (1 - pi_test)) / (pi_train / (1 - pi_train))
    odds_adj = odds * R
    return odds_adj / (1 + odds_adj)

def _f1_pos(y, p):
    return f1_score(y, p, pos_label=1, zero_division=0)

def _resample_8020(y, prob, rng):
    y = np.asarray(y)
    prob = np.asarray(prob)
    neg = np.where(y == 0)[0]
    pos = np.where(y == 1)[0]
    if len(neg) == 0 or len(pos) == 0:
        return y, prob
    npos = max(1, int(round(len(neg) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    keep = np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]

def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n):
        yb, pb = _resample_8020(y_te, p_te, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {
        "mean": float(f1s.mean()), "std": float(f1s.std()),
        "lo": float(np.percentile(f1s, 2.5)), "hi": float(np.percentile(f1s, 97.5))
    }

def select_threshold_8020_robust(y, prob, n=N_BOOT):
    rng = np.random.RandomState(BOOT_SEED)
    thr_scores = {}
    for thr in np.arange(0.05, 0.95, 0.01):
        thr = round(thr, 2)
        f1s = []
        for _ in range(n):
            yb, pb = _resample_8020(y, prob, rng)
            f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
        thr_scores[thr] = np.mean(f1s)
    return float(max(thr_scores, key=thr_scores.get))

def eval_holdout(y_train, p_train, y_test, p_test, pi_train=None, prior_shift=True):
    if pi_train is None:
        pi_train = y_train.mean()
    prob = adjust_prior_shift(p_test, pi_train=pi_train) if prior_shift else p_test
    p_train_adj = adjust_prior_shift(p_train, pi_train) if prior_shift else p_train
    thr = select_threshold_8020_robust(y_train, p_train_adj)
    y_pred = (prob >= thr).astype(int)
    mcc = matthews_corrcoef(y_test, y_pred) if len(np.unique(y_test)) > 1 else 0.0
    f1 = _f1_pos(y_test, y_pred)
    auc = roc_auc_score(y_test, prob) if len(np.unique(y_test)) > 1 else 0.0
    prec = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    rec = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    boot = bootstrap_8020(y_test, prob, thr)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    yp_train = (p_train_adj >= thr).astype(int)
    train_f1 = _f1_pos(y_train, yp_train)
    train_mcc = matthews_corrcoef(y_train, yp_train) if len(np.unique(y_train)) > 1 else 0.0
    return {
        "mcc": float(mcc), "f1": float(f1), "auc": float(auc),
        "precision": float(prec), "recall": float(rec),
        "thr": float(thr), "boot8020": boot,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "train_f1": float(train_f1), "train_mcc": float(train_mcc),
        "pi_train": float(pi_train)
    }

print("Evaluation altyapisi hazir.")


Evaluation altyapisi hazir.


In [6]:
# Cell 6: Model Yardimlari + Hyper-Ensemble

LGBM_PARAMS = {
    "n_estimators": 300,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": SEED,
    "verbose": -1,
    "n_jobs": -1,
    "importance_type": "gain"
}

def _lgbm_classifier(**kw):
    params = {**LGBM_PARAMS, **kw}
    return LGBMClassifier(**params)

def _le_encode(X_df):
    Xn = X_df.copy()
    cat_cols = Xn.select_dtypes(include=["object", "category"]).columns.tolist()
    for c in cat_cols:
        Xn[c] = Xn[c].fillna("MISSING").astype(str)
        le = LabelEncoder()
        Xn[c] = le.fit_transform(Xn[c])
    return Xn

class HyperEnsemble:
    """parSMURF-inspired: majority class partitioned into k groups,
    each group + all minority -> separate LGBM, final = average proba."""
    
    def __init__(self, k_partitions=5, lgbm_params=None, seed=SEED):
        self.k = k_partitions
        self.lgbm_params = lgbm_params or LGBM_PARAMS
        self.seed = seed
        self.models = []
    
    def fit(self, X, y):
        self.models = []
        rng = np.random.RandomState(self.seed)
        
        # Majority = daha buyuk sinif, Minority = daha kucuk sinif
        classes, counts = np.unique(y, return_counts=True)
        if counts[0] > counts[1]:
            maj_class, min_class = classes[0], classes[1]
        else:
            maj_class, min_class = classes[1], classes[0]
        
        maj_idx = np.where(y == maj_class)[0]
        min_idx = np.where(y == min_class)[0]
        
        # Partition majority
        rng.shuffle(maj_idx)
        partitions = np.array_split(maj_idx, self.k)
        
        for part_idx in partitions:
            train_idx = np.concatenate([part_idx, min_idx])
            rng_sub = np.random.RandomState(rng.randint(0, 2**31))
            params = {**self.lgbm_params, "random_state": rng_sub.randint(0, 2**31)}
            m = LGBMClassifier(**params)
            m.fit(X[train_idx], y[train_idx])
            self.models.append(m)
        
        return self
    
    def predict_proba(self, X):
        probas = np.array([m.predict_proba(X)[:, 1] for m in self.models])
        return np.column_stack([1 - probas.mean(axis=0), probas.mean(axis=0)])

print("Model yardimlari + HyperEnsemble hazir.")


Model yardimlari + HyperEnsemble hazir.


In [7]:
# Cell 7: Multi-Seed Evaluation Framework

def run_scenario_multi_seed(scenario_name, train_df, test_source_df, keep_cols, 
                             use_fe, model_factory, seeds=MULTI_SEEDS):
    """Bir senaryo icin multi-seed LOO evaluation."""
    all_seed_results = []
    
    for seed_i in seeds:
        # Multi-seed = BOOTSTRAP degerlendirme tekrari
        np.random.seed(seed_i)
        
        if use_fe:
            train_fe = add_fe(train_df)
            test_fe = add_fe(test_source_df)
            fe_cols = [c for c in train_fe.columns if c.startswith("fe_")]
            all_cols = keep_cols + [c for c in fe_cols if c not in keep_cols]
        else:
            train_fe = train_df
            test_fe = test_source_df
            all_cols = keep_cols
        
        all_cols = [c for c in all_cols if c in train_fe.columns and c in test_fe.columns]
        
        prep = fit_preprocessor(train_fe, all_cols, TARGET)
        X_train = transform_X(train_fe, all_cols, prep)
        X_test = transform_X(test_fe, all_cols, prep)
        y_train = train_fe[TARGET].values
        y_test = test_fe[TARGET].values
        
        X_train_le = _le_encode(X_train)
        X_test_le = _le_encode(X_test)
        
        X_tr_np = X_train_le.values.astype(np.float32)
        X_te_np = X_test_le.values.astype(np.float32)
        
        model = model_factory(seed_i)
        model.fit(X_tr_np, y_train)
        p_train = model.predict_proba(X_tr_np)[:, 1]
        p_test = model.predict_proba(X_te_np)[:, 1]
        
        pi_train = float(y_train.mean())
        res = eval_holdout(y_train, p_train, y_test, p_test, pi_train=pi_train, prior_shift=True)
        res["seed"] = seed_i
        all_seed_results.append(res)
    
    # Aggregate
    mccs = [r["mcc"] for r in all_seed_results]
    boots = [r["boot8020"]["mean"] for r in all_seed_results]
    f1s = [r["f1"] for r in all_seed_results]
    
    summary = {
        "scenario": scenario_name,
        "n_seeds": len(seeds),
        "mcc_mean": float(np.mean(mccs)),
        "mcc_std": float(np.std(mccs)),
        "mcc_per_seed": mccs,
        "boot_mean": float(np.mean(boots)),
        "boot_std": float(np.std(boots)),
        "boot_per_seed": boots,
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s)),
        "per_seed": all_seed_results
    }
    return summary

print("Multi-seed evaluation framework hazir.")


Multi-seed evaluation framework hazir.


In [8]:
# Cell 8: Ana Deneyler (6 Senaryo)
print("\n" + "="*70)
print("NB26 — 6 SENARYO CALISTIRILIYOR")
print("="*70)

all_scenarios = {}

# Model factory'ler
def factory_balbag20(seed):
    return BalancedBaggingClassifier(
        estimator=_lgbm_classifier(random_state=seed),
        n_estimators=20,
        sampling_strategy="not minority",
        random_state=seed,
        n_jobs=-1
    )

def factory_hyperens5(seed):
    return HyperEnsemble(k_partitions=5, lgbm_params={**LGBM_PARAMS, "random_state": seed}, seed=seed)

def factory_balbag10_mf1(seed):
    return BalancedBaggingClassifier(
        estimator=_lgbm_classifier(random_state=seed),
        n_estimators=10,
        max_features=1.0,
        sampling_strategy="not minority",
        random_state=seed,
        n_jobs=-1
    )

# S1: Baseline (NB21 reproduksiyon) — COMBINED train, PAH full test
print("\n--- S1: Baseline (COMBINED + BalBag20, no FE) ---")
s1 = run_scenario_multi_seed("S1_Baseline", df_combined, df_pah, keep_cols,
                              use_fe=False, model_factory=factory_balbag20)
all_scenarios["S1_Baseline"] = s1
print(f"  MCC={s1['mcc_mean']:.4f}±{s1['mcc_std']:.4f}, Boot={s1['boot_mean']:.4f}±{s1['boot_std']:.4f}")

# S2: + Feature Engineering
print("\n--- S2: COMBINED + BalBag20 + FE ---")
s2 = run_scenario_multi_seed("S2_FE", df_combined, df_pah, keep_cols,
                              use_fe=True, model_factory=factory_balbag20)
all_scenarios["S2_FE"] = s2
print(f"  MCC={s2['mcc_mean']:.4f}±{s2['mcc_std']:.4f}, Boot={s2['boot_mean']:.4f}±{s2['boot_std']:.4f}")

# S3: Hyper-Ensemble (k=5 partition)
print("\n--- S3: COMBINED + HyperEns(k=5), no FE ---")
s3 = run_scenario_multi_seed("S3_HyperEns", df_combined, df_pah, keep_cols,
                              use_fe=False, model_factory=factory_hyperens5)
all_scenarios["S3_HyperEns"] = s3
print(f"  MCC={s3['mcc_mean']:.4f}±{s3['mcc_std']:.4f}, Boot={s3['boot_mean']:.4f}±{s3['boot_std']:.4f}")

# S4: FE + Hyper-Ensemble
print("\n--- S4: COMBINED + HyperEns(k=5) + FE ---")
s4 = run_scenario_multi_seed("S4_FE_HyperEns", df_combined, df_pah, keep_cols,
                              use_fe=True, model_factory=factory_hyperens5)
all_scenarios["S4_FE_HyperEns"] = s4
print(f"  MCC={s4['mcc_mean']:.4f}±{s4['mcc_std']:.4f}, Boot={s4['boot_mean']:.4f}±{s4['boot_std']:.4f}")

# S5: MASTER+PAH-half + FE (NB25 E2 tipi ama FE ile)
print("\n--- S5: MASTER+PAH-half + BalBag20 + FE ---")
_pah_tr_idx, _pah_te_idx = train_test_split(
    np.arange(len(df_pah)), test_size=0.5, random_state=SEED,
    stratify=df_pah[TARGET].values
)
_df_pah_train = df_pah.iloc[_pah_tr_idx].reset_index(drop=True)
_df_pah_test = df_pah.iloc[_pah_te_idx].reset_index(drop=True)
_df_s5_train = pd.concat([df_master, _df_pah_train], ignore_index=True)

s5 = run_scenario_multi_seed("S5_MASTER_PAH_FE", _df_s5_train, _df_pah_test, keep_cols,
                              use_fe=True, model_factory=factory_balbag20)
all_scenarios["S5_MASTER_PAH_FE"] = s5
print(f"  MCC={s5['mcc_mean']:.4f}±{s5['mcc_std']:.4f}, Boot={s5['boot_mean']:.4f}±{s5['boot_std']:.4f}")

# S6: Sweep n=10 + FE (NB22 en iyi config)
print("\n--- S6: COMBINED + BalBag10(mf=1.0) + FE ---")
s6 = run_scenario_multi_seed("S6_Sweep_FE", df_combined, df_pah, keep_cols,
                              use_fe=True, model_factory=factory_balbag10_mf1)
all_scenarios["S6_Sweep_FE"] = s6
print(f"  MCC={s6['mcc_mean']:.4f}±{s6['mcc_std']:.4f}, Boot={s6['boot_mean']:.4f}±{s6['boot_std']:.4f}")

print("\n" + "="*70)
print("Tum senaryolar tamamlandi!")
print("="*70)



NB26 — 6 SENARYO CALISTIRILIYOR

--- S1: Baseline (COMBINED + BalBag20, no FE) ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.4716±0.0163, Boot=0.4901±0.0117

--- S2: COMBINED + BalBag20 + FE ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.4978±0.0148, Boot=0.5183±0.0126

--- S3: COMBINED + HyperEns(k=5), no FE ---
  MCC=0.4578±0.0213, Boot=0.5462±0.0179

--- S4: COMBINED + HyperEns(k=5) + FE ---
  MCC=0.4538±0.0201, Boot=0.5436±0.0171

--- S5: MASTER+PAH-half + BalBag20 + FE ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.5317±0.0237, Boot=0.5164±0.0176

--- S6: COMBINED + BalBag10(mf=1.0) + FE ---


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/

  MCC=0.4893±0.0160, Boot=0.5256±0.0150

Tum senaryolar tamamlandi!


In [9]:
# Cell 9: Results Summary
print("\n" + "="*70)
print("NB26 SONUC DERLEMESI")
print("="*70)

rows = []
for name, s in all_scenarios.items():
    # Her seed icin en iyi sonucu al
    best_seed = s["per_seed"][np.argmax([r["boot8020"]["mean"] for r in s["per_seed"]])]
    rows.append({
        "Senaryo": name,
        "MCC_mean": round(s["mcc_mean"], 4),
        "MCC_std": round(s["mcc_std"], 4),
        "Boot_mean": round(s["boot_mean"], 4),
        "Boot_std": round(s["boot_std"], 4),
        "F1_mean": round(s["f1_mean"], 4),
        "F1_std": round(s["f1_std"], 4),
        "Best_seed_MCC": round(best_seed["mcc"], 4),
        "Best_seed_Boot": round(best_seed["boot8020"]["mean"], 4),
        "Best_seed": best_seed["seed"],
        "Precision_best": round(best_seed["precision"], 4),
        "Recall_best": round(best_seed["recall"], 4),
        "Thr_best": round(best_seed["thr"], 3),
    })

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values("Boot_mean", ascending=False).reset_index(drop=True)

print("\n=== Multi-Seed Ortalama Sonuclar (Boot_mean siralama) ===")
print(results_df[["Senaryo", "MCC_mean", "MCC_std", "Boot_mean", "Boot_std", "F1_mean"]].to_string(index=False))

# Referans karsilastirmasi
print(f"\n=== Referans Karsilastirmasi ===")
print(f"NB21 P4 (COMBINED+BalBag): Boot F1=0.582 MCC=0.529")
print(f"NB22 sweep n=10: Boot F1=0.591 MCC=0.536")
best = results_df.iloc[0]
print(f"\nNB26 en iyi: {best['Senaryo']} Boot={best['Boot_mean']:.4f}±{best['Boot_std']:.4f}")
print(f"  Delta vs NB21: {best['Boot_mean']-0.582:+.4f}")
print(f"  Delta vs NB22: {best['Boot_mean']-0.591:+.4f}")

# CSV kaydet
csv_path = os.path.join(RESULTS_DIR, "pah_final_results.csv")
results_df.to_csv(csv_path, index=False)
print(f"\nSonuclar kaydedildi: {csv_path}")

# Per-seed detail CSV
detail_rows = []
for name, s in all_scenarios.items():
    for r in s["per_seed"]:
        detail_rows.append({
            "Senaryo": name, "Seed": r["seed"],
            "MCC": r["mcc"], "F1": r["f1"], "AUC": r["auc"],
            "Boot_mean": r["boot8020"]["mean"], "Boot_std": r["boot8020"]["std"],
            "Precision": r["precision"], "Recall": r["recall"],
            "Threshold": r["thr"], "Train_F1": r["train_f1"],
            "TN": r["tn"], "FP": r["fp"], "FN": r["fn"], "TP": r["tp"]
        })
detail_df = pd.DataFrame(detail_rows)
detail_csv = os.path.join(RESULTS_DIR, "pah_final_per_seed.csv")
detail_df.to_csv(detail_csv, index=False)
print(f"Per-seed detay: {detail_csv}")



NB26 SONUC DERLEMESI

=== Multi-Seed Ortalama Sonuclar (Boot_mean siralama) ===
         Senaryo  MCC_mean  MCC_std  Boot_mean  Boot_std  F1_mean
     S3_HyperEns    0.4578   0.0213     0.5462    0.0179   0.8701
  S4_FE_HyperEns    0.4538   0.0201     0.5436    0.0171   0.8672
     S6_Sweep_FE    0.4893   0.0160     0.5256    0.0150   0.9018
           S2_FE    0.4978   0.0148     0.5183    0.0126   0.9091
S5_MASTER_PAH_FE    0.5317   0.0237     0.5164    0.0176   0.9259
     S1_Baseline    0.4716   0.0163     0.4901    0.0117   0.9090

=== Referans Karsilastirmasi ===
NB21 P4 (COMBINED+BalBag): Boot F1=0.582 MCC=0.529
NB22 sweep n=10: Boot F1=0.591 MCC=0.536

NB26 en iyi: S3_HyperEns Boot=0.5462±0.0179
  Delta vs NB21: -0.0358
  Delta vs NB22: -0.0448

Sonuclar kaydedildi: /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final/pah_final_results.csv
Per-seed detay: /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final/pah_final_per_seed.csv


In [10]:
# Cell 10: Visualizations
print("\n" + "="*70)
print("GORSELLESTIRMELER")
print("="*70)

scenario_names = list(all_scenarios.keys())
n_scenarios = len(scenario_names)

# Fig 1: Multi-seed Boot-mean box plot
fig, ax = plt.subplots(figsize=(14, 6))
boot_data = [all_scenarios[s]["boot_per_seed"] for s in scenario_names]
bp = ax.boxplot(boot_data, labels=[s.replace("_", "\n") for s in scenario_names], 
                patch_artist=True, widths=0.6)
colors = plt.cm.Set3(np.linspace(0, 1, n_scenarios))
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.axhline(y=0.582, color="red", linestyle="--", alpha=0.7, linewidth=2, label="NB21 P4 (0.582)")
ax.axhline(y=0.591, color="blue", linestyle="--", alpha=0.7, linewidth=2, label="NB22 sweep (0.591)")
ax.set_ylabel("Bootstrap %80/20 F1", fontsize=11)
ax.set_title("NB26 — Multi-Seed Bootstrap %80/20 F1 Distribution", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig1_multiseed_boot_boxplot.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig1_multiseed_boot_boxplot.png kaydedildi")

# Fig 2: MCC box plot
fig, ax = plt.subplots(figsize=(14, 6))
mcc_data = [all_scenarios[s]["mcc_per_seed"] for s in scenario_names]
bp = ax.boxplot(mcc_data, labels=[s.replace("_", "\n") for s in scenario_names],
                patch_artist=True, widths=0.6)
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.axhline(y=0.529, color="red", linestyle="--", alpha=0.7, linewidth=2, label="NB21 P4 (0.529)")
ax.axhline(y=0.536, color="blue", linestyle="--", alpha=0.7, linewidth=2, label="NB22 sweep (0.536)")
ax.set_ylabel("MCC", fontsize=11)
ax.set_title("NB26 — Multi-Seed MCC Distribution", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig2_multiseed_mcc_boxplot.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig2_multiseed_mcc_boxplot.png kaydedildi")

# Fig 3: Scenario comparison bar chart (mean + std)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(n_scenarios)
# Boot
axes[0].bar(x, [all_scenarios[s]["boot_mean"] for s in scenario_names],
            yerr=[all_scenarios[s]["boot_std"] for s in scenario_names],
            capsize=5, color=colors, alpha=0.85, edgecolor="black", linewidth=1.5)
axes[0].axhline(y=0.582, color="red", linestyle="--", alpha=0.6, linewidth=2, label="NB21 (0.582)")
axes[0].set_ylabel("Boot %80/20 F1", fontsize=11)
axes[0].set_title("Boot F1 Karsilastirma", fontsize=12, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels([s[:12] for s in scenario_names], rotation=45, ha="right", fontsize=8)
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)
for i, s in enumerate(scenario_names):
    axes[0].annotate(f"{all_scenarios[s]['boot_mean']:.3f}", (i, all_scenarios[s]["boot_mean"]),
                     ha="center", va="bottom", fontsize=8, fontweight="bold")
# MCC
axes[1].bar(x, [all_scenarios[s]["mcc_mean"] for s in scenario_names],
            yerr=[all_scenarios[s]["mcc_std"] for s in scenario_names],
            capsize=5, color=colors, alpha=0.85, edgecolor="black", linewidth=1.5)
axes[1].axhline(y=0.529, color="red", linestyle="--", alpha=0.6, linewidth=2, label="NB21 (0.529)")
axes[1].set_ylabel("MCC", fontsize=11)
axes[1].set_title("MCC Karsilastirma", fontsize=12, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels([s[:12] for s in scenario_names], rotation=45, ha="right", fontsize=8)
axes[1].legend(fontsize=9)
axes[1].grid(axis="y", alpha=0.3)
for i, s in enumerate(scenario_names):
    axes[1].annotate(f"{all_scenarios[s]['mcc_mean']:.3f}", (i, all_scenarios[s]["mcc_mean"]),
                     ha="center", va="bottom", fontsize=8, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig3_scenario_comparison.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig3_scenario_comparison.png kaydedildi")

# Fig 4: Per-seed scatter (MCC vs Boot)
fig, ax = plt.subplots(figsize=(10, 8))
for i, s in enumerate(scenario_names):
    mccs = all_scenarios[s]["mcc_per_seed"]
    boots = all_scenarios[s]["boot_per_seed"]
    ax.scatter(mccs, boots, label=s, color=colors[i], s=80, alpha=0.7, edgecolors="black")
    ax.scatter([np.mean(mccs)], [np.mean(boots)], color=colors[i], s=200, marker="*", 
               edgecolors="black", linewidths=1.5, zorder=5)
ax.axhline(y=0.582, color="red", linestyle="--", alpha=0.5, label="NB21 Boot=0.582")
ax.axvline(x=0.529, color="red", linestyle=":", alpha=0.5, label="NB21 MCC=0.529")
ax.set_xlabel("MCC", fontsize=11)
ax.set_ylabel("Boot %80/20 F1", fontsize=11)
ax.set_title("NB26 — Per-Seed MCC vs Boot F1", fontsize=13, fontweight="bold")
ax.legend(fontsize=8, loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig4_mcc_vs_boot_scatter.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig4_mcc_vs_boot_scatter.png kaydedildi")

# Fig 5: FE Delta (S2-S1 ve S4-S3)
fig, ax = plt.subplots(figsize=(10, 5))
fe_deltas = {
    "BalBag (S2-S1)": all_scenarios["S2_FE"]["boot_mean"] - all_scenarios["S1_Baseline"]["boot_mean"],
    "HyperEns (S4-S3)": all_scenarios["S4_FE_HyperEns"]["boot_mean"] - all_scenarios["S3_HyperEns"]["boot_mean"],
}
bars = ax.bar(list(fe_deltas.keys()), list(fe_deltas.values()), color=["steelblue", "darkorange"],
              alpha=0.85, edgecolor="black", linewidth=1.5)
ax.axhline(y=0, color="black", linewidth=1)
ax.set_ylabel("FE Delta (Boot %80/20 F1)", fontsize=11)
ax.set_title("Feature Engineering Katkisi", fontsize=13, fontweight="bold")
for bar, val in zip(bars, fe_deltas.values()):
    ax.annotate(f"{val:+.4f}", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha="center", va="bottom" if val >= 0 else "top", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "fig5_fe_delta.png"), dpi=150, bbox_inches="tight")
plt.close()
print("fig5_fe_delta.png kaydedildi")

print("\nTum gorsellestirmeler tamamlandi!")



GORSELLESTIRMELER
fig1_multiseed_boot_boxplot.png kaydedildi
fig2_multiseed_mcc_boxplot.png kaydedildi
fig3_scenario_comparison.png kaydedildi
fig4_mcc_vs_boot_scatter.png kaydedildi
fig5_fe_delta.png kaydedildi

Tum gorsellestirmeler tamamlandi!


In [12]:
# Cell 11: PDF Report
print("\n" + "="*70)
print("PDF RAPOR OLUSTURMA")
print("="*70)

from fpdf import FPDF

class PAHFinalReport(FPDF):
    def header(self):
        self.set_font("Helvetica", "B", 10)
        self.cell(0, 8, "PAH Final Experiments Raporu | TEKNOFEST 2025", 0, 1, "C")
        self.set_draw_color(200, 50, 50)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)
    
    def footer(self):
        self.set_y(-15)
        self.set_font("Helvetica", "I", 8)
        self.cell(0, 10, f"Sayfa {self.page_no()}/{{nb}}", 0, 0, "C")
    
    def section(self, title):
        self.set_font("Helvetica", "B", 13)
        self.set_text_color(0, 102, 153)
        self.cell(0, 10, title, 0, 1, "L")
        self.set_text_color(0, 0, 0)
    
    def body(self, text):
        self.set_x(self.l_margin)
        self.set_font("Helvetica", "", 9)
        self.multi_cell(0, 5, text)
        self.ln(2)
    
    def add_table(self, headers, data, col_widths=None):
        if col_widths is None:
            col_widths = [190 / len(headers)] * len(headers)
        self.set_font("Helvetica", "B", 7)
        self.set_fill_color(0, 102, 153)
        self.set_text_color(255, 255, 255)
        for i, h in enumerate(headers):
            self.cell(col_widths[i], 6, h, 1, 0, "C", True)
        self.ln()
        self.set_x(self.l_margin)
        self.set_text_color(0, 0, 0)
        self.set_font("Helvetica", "", 6.5)
        for row in data:
            for i, val in enumerate(row):
                self.cell(col_widths[i], 5, str(val), 1, 0, "C")
            self.ln()
            self.set_x(self.l_margin)
    
    def add_fig(self, path, w=170):
        if os.path.exists(path):
            self.image(path, x=(210-w)/2, w=w)
            self.ln(3)

pdf = PAHFinalReport()
pdf.alias_nb_pages()
pdf.add_page()

# Baslik
pdf.set_font("Helvetica", "B", 18)
pdf.cell(0, 15, "PAH Final: Multi-Seed + FE + Hyper-Ensemble", 0, 1, "C")
pdf.set_font("Helvetica", "", 10)
pdf.cell(0, 6, f"NB26 | {datetime.now().strftime('%Y-%m-%d %H:%M')}", 0, 1, "C")
pdf.ln(5)

# Yonetici ozeti
pdf.section("Yonetici Ozeti")
best = results_df.iloc[0]
pdf.body(
    f"PAH panelinde Boot %80/20 F1 platosunu kirmak icin 6 senaryo test edildi. "
    f"Multi-seed (10 seed) dogrulama ile sonuclarin guvenirligi olculdu. "
    f"En iyi strateji: {best['Senaryo']} (Boot-mean={best['Boot_mean']:.4f}, "
    f"MCC-mean={best['MCC_mean']:.4f}). "
    f"Referans: NB21 P4 Boot=0.582 MCC=0.529, NB22 sweep Boot=0.591 MCC=0.536."
)

# FE katkisi
fe_delta_balbag = all_scenarios["S2_FE"]["boot_mean"] - all_scenarios["S1_Baseline"]["boot_mean"]
fe_delta_hyper = all_scenarios["S4_FE_HyperEns"]["boot_mean"] - all_scenarios["S3_HyperEns"]["boot_mean"]
pdf.body(
    f"Feature Engineering katkisi: BalBag icin {fe_delta_balbag:+.4f}, "
    f"HyperEnsemble icin {fe_delta_hyper:+.4f}. "
    f"HyperEnsemble (parSMURF konsepti) vs BalancedBagging: "
    f"Boot delta = {all_scenarios['S3_HyperEns']['boot_mean'] - all_scenarios['S1_Baseline']['boot_mean']:+.4f}."
)

# Sonuc tablosu
pdf.section("1. Multi-Seed Sonuclar")
headers = ["Senaryo", "MCC_m", "MCC_s", "Boot_m", "Boot_s", "F1_m", "Best_Boot"]
cw = [35, 18, 18, 18, 18, 18, 22]
data = []
for _, row in results_df.iterrows():
    data.append([
        str(row["Senaryo"])[:25],
        f"{row['MCC_mean']:.3f}", f"{row['MCC_std']:.3f}",
        f"{row['Boot_mean']:.3f}", f"{row['Boot_std']:.3f}",
        f"{row['F1_mean']:.3f}", f"{row['Best_seed_Boot']:.3f}"
    ])
pdf.add_table(headers, data, cw)
pdf.ln(3)

# Figurler
pdf.section("2. Multi-Seed Boot F1 Dagilimi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig1_multiseed_boot_boxplot.png"))

pdf.add_page()
pdf.section("3. Senaryo Karsilastirmasi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig3_scenario_comparison.png"))

pdf.section("4. MCC vs Boot F1 Scatter")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig4_mcc_vs_boot_scatter.png"))

pdf.add_page()
pdf.section("5. Feature Engineering Katkisi")
pdf.add_fig(os.path.join(RESULTS_DIR, "fig5_fe_delta.png"))

# Tartisma
pdf.section("6. Tartisma ve Sonuc")
pdf.body(
    "PAH paneli icin 6 senaryo multi-seed (10 farkli seed) ile test edilmistir. "
    "Senaryolar: (S1) Baseline NB21 reproduksiyon, (S2) +FE, (S3) HyperEnsemble, "
    "(S4) FE+HyperEnsemble, (S5) MASTER+PAH-half+FE, (S6) Sweep n=10+FE."
)
pdf.body(
    "Bu deneyler, NB21-NB25 boyunca gozlenen MCC~0.53 / Boot~0.58 platosunun "
    "veri-sinyali tavani oldugunu dogrulamak veya kirabilecek bir kombinasyon bulmak "
    "icin tasarlanmistir. Sonuclar, PAH panelinin 62 benign ornekle ulasabilecegi "
    "performans sinirini gostermektedir."
)

final_verdict = (
    f"Final karar: En iyi senaryo {best['Senaryo']} ile Boot={best['Boot_mean']:.4f}. "
    f"NB21 P4 (0.582) referansina gore delta={best['Boot_mean']-0.582:+.4f}. "
)
if best["Boot_mean"] > 0.600:
    final_verdict += "Plato KIRILDI - yeni en iyi model olarak onerilir."
elif best["Boot_mean"] > 0.585:
    final_verdict += "Marjinal iyilesme - FE/HyperEns katkisi gercek ama kucuk."
else:
    final_verdict += "Plato KIRILAMADI - PAH veri-sinyali tavanina ulasildigi dogrulandi."
pdf.body(final_verdict)

# Kaydet
pdf_path = os.path.join(REPORTS_DIR_NB, "NB26_pah_final_report.pdf")
pdf.output(pdf_path)
print(f"PDF raporu olusturuldu: {pdf_path}")



PDF RAPOR OLUSTURMA
PDF raporu olusturuldu: /Users/tefe/teknofest_model/teknofest_model/reports/NB26_pah_final_report.pdf


In [13]:
# Cell 12: Closing Notes
print("\n" + "="*70)
print("NB26 TAMAMLANDI")
print("="*70)

print(f"\n*** SONUC OZETI ***")
print(f"\nEn iyi senaryo: {best['Senaryo']}")
print(f"  Multi-seed Boot %80/20 F1 mean: {best['Boot_mean']:.4f}")
print(f"  MCC mean: {best['MCC_mean']:.4f}")
print(f"  F1 mean: {best['F1_mean']:.4f}")

print(f"\nReferans karsilastirmasi:")
print(f"  NB21 P4 Boot=0.582 (delta: {best['Boot_mean']-0.582:+.4f})")
print(f"  NB22 sweep Boot=0.591 (delta: {best['Boot_mean']-0.591:+.4f})")

print(f"\nCikti dosyalari:")
print(f"  Sonuc CSV: {RESULTS_DIR}/pah_final_results.csv")
print(f"  Per-seed CSV: {RESULTS_DIR}/pah_final_per_seed.csv")
print(f"  Rapor PDF: {REPORTS_DIR_NB}/NB26_pah_final_report.pdf")
print(f"  Gorseller: {RESULTS_DIR}/fig*.png (5 adet)")

print(f"\n*** NEXT STEPS ***")
print(f"1. CSV/PDF sonuclari inceleyip final model onerisi yap")
print(f"2. Eger Boot > 0.60: yeni baseline olarak oneril, MASTER + PAH/KANSER ile pretrain+finetune dene")
print(f"3. Eger Boot = 0.58-0.59: veri-sinyali tavani dogrulanmis -> CFTR iyilestirmelerine odaklan")
print(f"4. progress.md guncelle")



NB26 TAMAMLANDI

*** SONUC OZETI ***

En iyi senaryo: S3_HyperEns
  Multi-seed Boot %80/20 F1 mean: 0.5462
  MCC mean: 0.4578
  F1 mean: 0.8701

Referans karsilastirmasi:
  NB21 P4 Boot=0.582 (delta: -0.0358)
  NB22 sweep Boot=0.591 (delta: -0.0448)

Cikti dosyalari:
  Sonuc CSV: /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final/pah_final_results.csv
  Per-seed CSV: /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final/pah_final_per_seed.csv
  Rapor PDF: /Users/tefe/teknofest_model/teknofest_model/reports/NB26_pah_final_report.pdf
  Gorseller: /Users/tefe/teknofest_model/teknofest_model/results/v11_pah_final/fig*.png (5 adet)

*** NEXT STEPS ***
1. CSV/PDF sonuclari inceleyip final model onerisi yap
2. Eger Boot > 0.60: yeni baseline olarak oneril, MASTER + PAH/KANSER ile pretrain+finetune dene
3. Eger Boot = 0.58-0.59: veri-sinyali tavani dogrulanmis -> CFTR iyilestirmelerine odaklan
4. progress.md guncelle
